# Shortest path: optimization versus algorithms

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/networks/shortest-path-optimization-vs-algorithms.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/networks/shortest-path-optimization-vs-algorithms.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

Colab already provides the general-purpose scientific libraries. This cell ensures the tested Pyomo and HiGHS versions; pip keeps an already-installed matching version. It does not replace Colab's NumPy, pandas or Matplotlib just to match the maintenance environment.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
from importlib.util import find_spec
import subprocess
import sys

required_packages = {
    'highspy': 'highspy',
    'networkx': 'networkx',
    'pandas': 'pandas',
    'pyomo': 'pyomo',
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = "appsi_highs"
assert pyo.SolverFactory(SOLVER).available(exception_flag=False)


In [ ]:
import networkx as nx
graph = nx.DiGraph()
graph.add_weighted_edges_from([('A', 'B', 10), ('A', 'D', 5), ('B', 'C', 1), ('B', 'D', 2), ('C', 'E', 4), ('D', 'B', 3), ('D', 'C', 9), ('D', 'E', 2), ('E', 'A', 7), ('E', 'C', 6)])


## The same question, two formulations

Send one unit of flow out of A and into C. At every other node, inflow equals outflow. Minimize the sum of length times flow. The LP is continuous: for this network structure an integral optimal extreme point exists. Ties may nevertheless allow fractional optimal flows, so compare objective values rather than assuming every returned flow encodes a unique path.


In [ ]:
def shortest_path_model(graph, source, target):
    if source not in graph or target not in graph or source == target:
        raise ValueError("Use two distinct nodes in the graph")
    if not nx.has_path(graph, source, target):
        raise ValueError("No directed path connects the requested nodes")
    if any(d['weight'] < 0 for _, _, d in graph.edges(data=True)):
        raise ValueError("This teaching comparison assumes nonnegative lengths")
    m = pyo.ConcreteModel("Unit-flow shortest path")
    m.N = pyo.Set(initialize=list(graph.nodes))
    m.A = pyo.Set(dimen=2, initialize=list(graph.edges))
    m.x = pyo.Var(m.A, domain=pyo.NonNegativeReals)
    m.cost = pyo.Objective(expr=pyo.quicksum(graph[u][v]['weight'] * m.x[u,v] for u,v in m.A))
    @m.Constraint(m.N)
    def flow(m, node):
        outgoing = pyo.quicksum(m.x[node,v] for v in graph.successors(node))
        incoming = pyo.quicksum(m.x[u,node] for u in graph.predecessors(node))
        return outgoing - incoming == int(node == source) - int(node == target)
    return m


In [ ]:
model = shortest_path_model(graph, 'A', 'C')
result = pyo.SolverFactory(SOLVER).solve(model)
assert_optimal_termination(result)
route = nx.shortest_path(graph, 'A', 'C', weight='weight')
distance = nx.path_weight(graph, route, weight='weight')
print('LP objective:', pyo.value(model.cost), 'NetworkX distance:', distance)
assert abs(pyo.value(model.cost) - distance) < 1e-7
assert distance == 9


## Reproducible timing comparison
Every method receives the same weighted graph. We use a fixed seed and modest sizes: these measurements illustrate overhead and scaling, not a universal performance ranking. NetworkX implements a specialized algorithm; Pyomo expresses a more general optimization model.


In [ ]:
import random
from time import perf_counter
import pandas as pd
rows = []
for size in [20, 50, 100]:
    undirected = nx.connected_watts_strogatz_graph(size, 4, 0.3, seed=42)
    rng = random.Random(42)
    nx.set_edge_attributes(undirected, {e:rng.randint(1,20) for e in undirected.edges}, 'weight')
    g = undirected.to_directed()
    start = perf_counter()
    lp = shortest_path_model(g, 0, size-1)
    result = pyo.SolverFactory(SOLVER).solve(lp)
    assert_optimal_termination(result)
    lp_seconds = perf_counter() - start
    start = perf_counter()
    distance = nx.shortest_path_length(g, 0, size-1, weight='weight')
    nx_seconds = perf_counter() - start
    assert abs(pyo.value(lp.cost)-distance) < 1e-6
    rows.append((size, lp_seconds, nx_seconds, distance))
pd.DataFrame(rows, columns=['nodes','LP build + solve (s)','NetworkX (s)','distance'])


## Extend the comparison
Add a constraint limiting a particular resource along the route. Does the specialized shortest-path algorithm still solve the altered problem? Distinguish a changed problem from a faster implementation of the same problem.
